In [0]:
from pyspark.sql.functions import col, count, when, lit, current_timestamp

bronze_df = spark.table("workspace.default.bronze_pet_store_records")

display(bronze_df)

In [0]:
total_rows = bronze_df.count()
total_columns = len(bronze_df.columns)

print("Total rows:", total_rows)
print("Total columns:", total_columns)
bronze_df.printSchema()

In [0]:
dq_checks = []

dq_checks.append(("total_rows", total_rows))
dq_checks.append(("null_product_id", bronze_df.filter(col("product_id").isNull()).count()))
dq_checks.append(("null_product_category", bronze_df.filter(col("product_category").isNull()).count()))
dq_checks.append(("null_vendor_id", bronze_df.filter(col("vendor_id").isNull()).count()))
dq_checks.append(("duplicate_product_id", bronze_df.groupBy("product_id").count().filter(col("count") > 1).count()))
dq_checks.append(("invalid_sales", bronze_df.filter((col("sales").isNull()) | (col("sales") < 0)).count()))
dq_checks.append(("invalid_price", bronze_df.filter((col("price").isNull()) | (col("price") <= 0)).count()))
dq_checks.append(("invalid_rating", bronze_df.filter((col("rating") < 1) | (col("rating") > 10)).count()))
dq_checks.append(("invalid_rebuy", bronze_df.filter(~col("re_buy").isin(0, 1)).count()))
dq_checks.append(("invalid_vap", bronze_df.filter(~col("VAP").isin(0, 1)).count()))

In [0]:
dq_df = spark.createDataFrame(dq_checks, ["check_name", "failed_record_count"])

dq_df = (
    dq_df
    .withColumn("table_name", lit("bronze_pet_store_records"))
    .withColumn("pipeline_layer", lit("bronze"))
    .withColumn("check_timestamp", current_timestamp())
)

display(dq_df)

In [0]:
dq_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dq_pet_store_records")

In [0]:
dq_check = spark.table("workspace.default.dq_pet_store_records")

display(dq_check)